In [88]:
import numpy as np

from pathlib import Path
from ngio import create_empty_ome_zarr, open_ome_zarr_container
from tifffile import imread

from zarr import open_group

In [2]:
training_zarr_path = Path("/Users/eglijan/code/fmi-basel/ggrossha_SWI/sandbox/training_data.zarr")

### Legacy data (tif)

In [7]:
scratch = Path("/Volumes/tachyon/groups/scratch")

In [51]:
data_dirs = [
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/manually_selected_for_v3",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/samples_20180601",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/samples_20180601_early",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/samples_20180601_extra",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/samples_20190719",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_hausyann/samples_20190719_extra",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_meeumilo/2020-03-06_finished",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_meeumilo/2020-03-13_finished",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_meeumilo/2020-07-08_ManualSelection",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_meeumilo/manually-selected_finished",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_moraluca/Chamber_200131_lin29A",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_moraluca/Chamber_200207_lin29Av2",
    # scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_moraluca/Chamber_200214_test_lin29A",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_smita",
    scratch / "gmicro_share/rempmark/worm_segmentation_samples/samples_vandmari/IX81exp1",
]

In [52]:
assert all((dir / "img").exists() for dir in data_dirs)
assert all((dir / "segm").exists() for dir in data_dirs)


In [53]:
# Get list of files with same name in img and segm folders
filenames = {}
for d in data_dirs:
    filenames[d] = []
    segm_files = sorted((d / "segm").glob("*.tif"))
    for f in segm_files:
        img_file = d / "img" / f.name
        if img_file.exists():
            filenames[d].append(f.name)
    print(f"{len(filenames[d])} image-segmentation pairs found in {d.name}")

29 image-segmentation pairs found in manually_selected_for_v3
100 image-segmentation pairs found in samples_20180601
50 image-segmentation pairs found in samples_20180601_early
9 image-segmentation pairs found in samples_20180601_extra
100 image-segmentation pairs found in samples_20190719
9 image-segmentation pairs found in samples_20190719_extra
115 image-segmentation pairs found in 2020-03-06_finished
100 image-segmentation pairs found in 2020-03-13_finished
112 image-segmentation pairs found in 2020-07-08_ManualSelection
55 image-segmentation pairs found in manually-selected_finished
100 image-segmentation pairs found in Chamber_200131_lin29A
88 image-segmentation pairs found in Chamber_200207_lin29Av2
73 image-segmentation pairs found in samples_smita
271 image-segmentation pairs found in IX81exp1


In [57]:
num_samples = int(np.sum([len(l) for l in filenames.values()]))
num_samples

1211

In [55]:
for d in data_dirs:
    img_files = sorted((d / "img").glob("*.tif"))
    segm_files = sorted((d / "segm").glob("*.tif"))
    print(f"{len(img_files)}\t{len(segm_files)}\tshape: {imread(img_files[0]).shape}\tin {d.name}")


29	29	shape: (1, 1024, 1016)	in manually_selected_for_v3
100	100	shape: (1001, 1001)	in samples_20180601
50	50	shape: (1001, 1001)	in samples_20180601_early
9	9	shape: (1024, 1016)	in samples_20180601_extra
100	100	shape: (1024, 1016)	in samples_20190719
9	9	shape: (1024, 1016)	in samples_20190719_extra
115	115	shape: (1024, 1016)	in 2020-03-06_finished
100	100	shape: (1024, 1016)	in 2020-03-13_finished
112	112	shape: (1, 1024, 1016)	in 2020-07-08_ManualSelection
55	55	shape: (1, 1024, 1016)	in manually-selected_finished
100	100	shape: (1, 1200, 1200)	in Chamber_200131_lin29A
100	88	shape: (1, 1200, 1200)	in Chamber_200207_lin29Av2
73	73	shape: (1, 1024, 1016)	in samples_smita
271	271	shape: (1, 1000, 1000)	in IX81exp1


In [ ]:
# Create OME-Zarr with num_samples, 1200, 1200 shape,
# chunked (1, 1200, 1200), xy_pixelsize=1.0, levels=1




In [82]:
training_zarr = create_empty_ome_zarr(
    store=training_zarr_path,
    shape=(num_samples, 1200, 1200),
    chunks=(1, 1200, 1200),
    xy_pixelsize=1.0,
    levels=["s0"],
    overwrite=True,
    dimension_separator=".",
    axes_names=["t", "y", "x"],
)

In [83]:
labels_data = training_zarr.derive_label(
    name="mask",
)

In [84]:
labels_data.zarr_array

<zarr.core.Array '/labels/mask/0' (1211, 1200, 1200) uint32>

In [ ]:
# then loop over files and copy data into zarr

image_data = training_zarr.get_image()


In [85]:
count = 0

for d in filenames:
    for fname in filenames[d]:
        img_path = d / "img" / fname
        segm_path = d / "segm" / fname

        img = imread(img_path).squeeze()
        segm = imread(segm_path).squeeze()

        image_data.zarr_array[count, 0:img.shape[0], 0:img.shape[1]] = img
        labels_data.zarr_array[count, 0:segm.shape[0], 0:segm.shape[1]] = segm

        count += 1

### New data (zarr)

In [34]:
zarr_dirs = [
    scratch / "gmicro_prefect/ggrossha/ancneagu/ggrossha_GRH1/processed_data/20250328_grh-1_reporter_new_RNAi/train_data",
    scratch / "gmicro_prefect/ggrossha/ggrossha_SWI-Template/processed_data/lin29a_lin29b-model/train_data",
    scratch / "gmicro_prefect/ggrossha/woelmich/20240105_grh1_23scr/training_data",
]

In [35]:
zarr_names = {
    "train": "worm-segmentation-train-data.zarr",
    "val": "worm-segmentation-val-data.zarr",
}

In [37]:
for d in zarr_dirs:
    zarr_group = open_group(d / zarr_names["train"], mode="r")
    print(zarr_group["x/0"])
    print(zarr_group["y/0"])
    val_group = open_group(d / zarr_names["val"], mode="r")
    print(val_group["x/0"])
    print(val_group["y/0"])

<zarr.core.Array '/x/0' (518, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (518, 1024, 1024) uint16 read-only>
<zarr.core.Array '/x/0' (59, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (59, 1024, 1024) uint16 read-only>
<zarr.core.Array '/x/0' (11, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (11, 1024, 1024) uint16 read-only>
<zarr.core.Array '/x/0' (1, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (1, 1024, 1024) uint16 read-only>
<zarr.core.Array '/x/0' (1932, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (1932, 1024, 1024) uint16 read-only>
<zarr.core.Array '/x/0' (217, 25, 1024, 1024) uint16 read-only>
<zarr.core.Array '/y/0' (217, 1024, 1024) uint16 read-only>


In [86]:
num_zarr_samples = int(np.sum([
    open_group(d / zarr_names["train"], mode="r")["x/0"].shape[0] +
    open_group(d / zarr_names["val"], mode="r")["x/0"].shape[0] for d in zarr_dirs
]))

In [87]:
num_zarr_samples

2738

In [99]:
np.sum((image_data.zarr_array.shape, (num_zarr_samples, 0, 0)), axis=0)

array([3949, 1200, 1200])

In [102]:
new_size = (image_data.zarr_array.shape[0] + num_zarr_samples,) + image_data.zarr_array.shape[1:]

In [103]:
image_data.zarr_array.resize(new_size)
labels_data.zarr_array.resize(new_size)

In [105]:
num_samples, num_zarr_samples, num_samples + num_zarr_samples

(1211, 2738, 3949)

In [147]:
count = num_samples  # offset for appending to zarr

for idx, d in enumerate(zarr_dirs):
    train_group = open_group(d / zarr_names["train"], mode="r")
    if idx == 2:  # special-case zarr data 'woelmich'
        train_group["x/0"].chunk_store._dimension_separator = "/"
        train_group["y/0"].chunk_store._dimension_separator = "/"
    train_x = train_group["x/0"]
    train_y = train_group["y/0"]
    print(f"Adding {train_x.shape[0]} samples from {d / zarr_names['train']}")
    for i in range(train_x.shape[0]):
        train_x_mip = np.min(train_x[i], axis=0)
        image_data.zarr_array[count, 0:train_x_mip.shape[0], 0:train_x_mip.shape[1]] = train_x_mip
        labels_data.zarr_array[count, 0:train_y.shape[1], 0:train_y.shape[2]] = train_y[i]
        count += 1

    val_group = open_group(d / zarr_names["val"], mode="r")
    if idx == 2:  # special-case zarr data 'woelmich'
        val_group["x/0"].chunk_store._dimension_separator = "/"
        val_group["y/0"].chunk_store._dimension_separator = "/"
    val_x = val_group["x/0"]
    val_y = val_group["y/0"]
    print(f"Adding {val_x.shape[0]} samples from {d / zarr_names['val']}")
    for i in range(val_x.shape[0]):
        val_x_mip = np.min(val_x[i], axis=0)
        image_data.zarr_array[count, 0:val_x_mip.shape[0], 0:val_x_mip.shape[1]] = val_x_mip
        labels_data.zarr_array[count, 0:val_y.shape[1], 0:val_y.shape[2]] = val_y[i]
        count += 1

print("Done.")

Adding 518 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/ancneagu/ggrossha_GRH1/processed_data/20250328_grh-1_reporter_new_RNAi/train_data/worm-segmentation-train-data.zarr
Adding 59 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/ancneagu/ggrossha_GRH1/processed_data/20250328_grh-1_reporter_new_RNAi/train_data/worm-segmentation-val-data.zarr
Adding 11 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/ggrossha_SWI-Template/processed_data/lin29a_lin29b-model/train_data/worm-segmentation-train-data.zarr
Adding 1 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/ggrossha_SWI-Template/processed_data/lin29a_lin29b-model/train_data/worm-segmentation-val-data.zarr
Adding 1932 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/woelmich/20240105_grh1_23scr/training_data/worm-segmentation-train-data.zarr
Adding 217 samples from /Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/woelmich/20240

In [131]:
test_dir = zarr_dirs[2]
test_dir

PosixPath('/Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/woelmich/20240105_grh1_23scr/training_data')

In [143]:
test_group = open_group(test_dir / zarr_names["train"], mode="r")

In [144]:
test_array = test_group["x/0"]

In [145]:
test_array.chunk_store._dimension_separator = "/"

In [146]:
test_array[123, :, 100, 100]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0], dtype=uint16)

In [133]:
test_group["x/0"].chunk_store._dimension_separator = "/"

In [138]:
test_group["x/0"][123, :, 100, 100]

array([6653, 6667, 6942, 6830, 7078, 7141, 7052, 6997, 6568, 6834, 7007,
       7913, 8505, 8694, 8842, 7708, 7032, 5554, 5116, 5685, 5418, 5310,
       5602, 5986, 6609], dtype=uint16)

In [142]:
np.min(test_group["x/0"][123], axis=0)

array([[6510, 6510, 6510, ..., 6348, 6290, 6225],
       [6510, 6630, 6703, ..., 6267, 6464, 6306],
       [6583, 6649, 6739, ..., 6507, 6568, 6437],
       ...,
       [6154, 6310, 6218, ..., 3479, 3567, 3651],
       [6360, 6259, 6153, ..., 3567, 3806, 3781],
       [6078, 5990, 5880, ..., 3746, 3645, 3923]],
      shape=(1024, 1024), dtype=uint16)